# 3. Model Training
## Testing All Feature × Model Combinations (6 total)

In [11]:
import sys
sys.path.append('..')

import joblib
import pandas as pd
import matplotlib.pyplot as plt
from src.models.baseline import BaselineModels
from src.models.traditional_ml import TraditionalMLModels
from src.evaluation import ModelEvaluator
from src.utils import save_model

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## Load All Features

In [12]:
# Load TF-IDF features
X_train_tfidf = joblib.load('../data/processed/X_train_tfidf.pkl')
X_test_tfidf = joblib.load('../data/processed/X_test_tfidf.pkl')

# Load Bag of Words features
X_train_bow = joblib.load('../data/processed/X_train_count.pkl')
X_test_bow = joblib.load('../data/processed/X_test_count.pkl')

# Load labels
y_train = joblib.load('../data/processed/y_train.pkl')
y_test = joblib.load('../data/processed/y_test.pkl')

print(f"TF-IDF - Train: {X_train_tfidf.shape}, Test: {X_test_tfidf.shape}")
print(f"BoW    - Train: {X_train_bow.shape}, Test: {X_test_bow.shape}")
print(f"Labels - Train: {y_train.shape}, Test: {y_test.shape}")

TF-IDF - Train: (120000, 5000), Test: (7600, 5000)
BoW    - Train: (120000, 5000), Test: (7600, 5000)
Labels - Train: (120000,), Test: (7600,)


## Initialize Evaluator

In [13]:
evaluator = ModelEvaluator()

## Test All Combinations

In [16]:
# Define features and models to test
features = {
    'BoW': (X_train_bow, X_test_bow),
    'TF-IDF': (X_train_tfidf, X_test_tfidf)
}

models_config = {
    'Naive Bayes': BaselineModels.get_naive_bayes(),
    'Logistic Regression': BaselineModels.get_logistic_regression(),
    'Linear SVM': TraditionalMLModels.get_svm()
}

# Store all results
all_results = []

# Test all combinations
for feature_name, (X_train, X_test) in features.items():
    print(f"\n{'='*60}")
    print(f"Feature: {feature_name}")
    print('='*60)
    
    for model_name, model in models_config.items():
        print(f"\nTraining {model_name} with {feature_name}...")
        
        # Train and evaluate
        metrics, predictions = evaluator.train_and_evaluate(
            model, X_train, y_train, X_test, y_test, 
            f"{model_name} ({feature_name})"
        )
        
        # Add feature type to metrics
        metrics['feature_type'] = feature_name
        metrics['model_only'] = model_name
        
        all_results.append(metrics)
        
        # Print results
        print(f"  Accuracy:         {metrics['accuracy']:.4f}")
        print(f"  Precision (Macro): {metrics['precision_macro']:.4f}")
        print(f"  Recall (Macro):    {metrics['recall_macro']:.4f}")
        print(f"  F1-Score (Macro):  {metrics['f1_macro']:.4f}")
        print(f"  F1-Score (Weighted): {metrics['f1_weighted']:.4f}")
        print(f"  Training Time:     {metrics['train_time']:.2f}s")
        
        # Save model
        model_filename = f"../models/{model_name.lower().replace(' ', '_')}_{feature_name.lower().replace('-', '')}.pkl"
        save_model(model, model_filename)

print(f"\n{'='*60}")
print("All 6 combinations trained successfully!")
print('='*60)


Feature: BoW

Training Naive Bayes with BoW...
  Accuracy:         0.8909
  Precision (Macro): 0.8904
  Recall (Macro):    0.8909
  F1-Score (Macro):  0.8906
  F1-Score (Weighted): 0.8906
  Training Time:     0.02s
Model saved to ../models/naive_bayes_bow.pkl

Training Logistic Regression with BoW...
  Accuracy:         0.8954
  Precision (Macro): 0.8952
  Recall (Macro):    0.8954
  F1-Score (Macro):  0.8952
  F1-Score (Weighted): 0.8952
  Training Time:     2.31s
Model saved to ../models/logistic_regression_bow.pkl

Training Linear SVM with BoW...
  Accuracy:         0.9034
  Precision (Macro): 0.9033
  Recall (Macro):    0.9034
  F1-Score (Macro):  0.9032
  F1-Score (Weighted): 0.9032
  Training Time:     7.53s
Model saved to ../models/linear_svm_bow.pkl

Feature: TF-IDF

Training Naive Bayes with TF-IDF...
  Accuracy:         0.8903
  Precision (Macro): 0.8897
  Recall (Macro):    0.8903
  F1-Score (Macro):  0.8899
  F1-Score (Weighted): 0.8899
  Training Time:     0.02s
Model sav

## Results Comparison Table

In [17]:
# Create results dataframe
results_df = pd.DataFrame(all_results)

# Reorder columns for better readability
column_order = ['model_name', 'feature_type', 'model_only', 'accuracy', 
                'precision_macro', 'recall_macro', 'f1_macro', 'f1_weighted',
                'train_time', 'predict_time']
results_df = results_df[column_order]

# Save full results
results_df.to_csv('../results/comparison.csv', index=False)

print("\nFull Results Table:")
print(results_df.to_string())


Full Results Table:
                     model_name feature_type           model_only  accuracy  precision_macro  recall_macro  f1_macro  f1_weighted  train_time  predict_time
0             Naive Bayes (BoW)          BoW          Naive Bayes  0.890921         0.890395      0.890921  0.890555     0.890555    0.020517      0.000999
1     Logistic Regression (BoW)          BoW  Logistic Regression  0.895395         0.895180      0.895395  0.895237     0.895237    2.314186      0.001003
2              Linear SVM (BoW)          BoW           Linear SVM  0.903421         0.903277      0.903421  0.903237     0.903237    7.532522      0.001000
3          Naive Bayes (TF-IDF)       TF-IDF          Naive Bayes  0.890263         0.889725      0.890263  0.889920     0.889920    0.018510      0.000998
4  Logistic Regression (TF-IDF)       TF-IDF  Logistic Regression  0.905789         0.905525      0.905789  0.905570     0.905570    2.055209      0.000000
5           Linear SVM (TF-IDF)       TF-ID

## Best Model per Feature Type

In [18]:
print("\nBest Model for Each Feature Type:")
print("="*60)

for feature_type in ['BoW', 'TF-IDF']:
    subset = results_df[results_df['feature_type'] == feature_type]
    best_idx = subset['accuracy'].idxmax()
    best = subset.loc[best_idx]
    
    print(f"\n{feature_type}:")
    print(f"  Best Model: {best['model_only']}")
    print(f"  Accuracy: {best['accuracy']:.4f}")
    print(f"  F1-Score: {best['f1_macro']:.4f}")


Best Model for Each Feature Type:

BoW:
  Best Model: Linear SVM
  Accuracy: 0.9034
  F1-Score: 0.9032

TF-IDF:
  Best Model: Linear SVM
  Accuracy: 0.9075
  F1-Score: 0.9073


## Overall Best Model

In [19]:
best_overall_idx = results_df['accuracy'].idxmax()
best_overall = results_df.loc[best_overall_idx]

print("\nOverall Best Model:")
print("="*60)
print(f"Model: {best_overall['model_only']}")
print(f"Feature: {best_overall['feature_type']}")
print(f"Accuracy: {best_overall['accuracy']:.4f}")
print(f"F1-Score: {best_overall['f1_macro']:.4f}")
print(f"Precision: {best_overall['precision_macro']:.4f}")
print(f"Recall: {best_overall['recall_macro']:.4f}")
print(f"Training Time: {best_overall['train_time']:.2f}s")


Overall Best Model:
Model: Linear SVM
Feature: TF-IDF
Accuracy: 0.9075
F1-Score: 0.9073
Precision: 0.9074
Recall: 0.9075
Training Time: 2.88s


## Feature Type Comparison

In [20]:
# Average performance by feature type
feature_comparison = results_df.groupby('feature_type')[['accuracy', 'f1_macro', 'train_time']].mean()

print("\nAverage Performance by Feature Type:")
print(feature_comparison)


Average Performance by Feature Type:
              accuracy  f1_macro  train_time
feature_type                                
BoW           0.896579  0.896343    3.289075
TF-IDF        0.901184  0.900939    1.652308


## Model Comparison Across Features

In [21]:
# Pivot table for easier comparison
pivot_accuracy = results_df.pivot(index='model_only', columns='feature_type', values='accuracy')
pivot_f1 = results_df.pivot(index='model_only', columns='feature_type', values='f1_macro')

print("\nAccuracy Comparison:")
print(pivot_accuracy)

print("\nF1-Score Comparison:")
print(pivot_f1)


Accuracy Comparison:
feature_type              BoW    TF-IDF
model_only                             
Linear SVM           0.903421  0.907500
Logistic Regression  0.895395  0.905789
Naive Bayes          0.890921  0.890263

F1-Score Comparison:
feature_type              BoW    TF-IDF
model_only                             
Linear SVM           0.903237  0.907328
Logistic Regression  0.895237  0.905570
Naive Bayes          0.890555  0.889920


## Summary

In [22]:
print("\n" + "="*60)
print("EXPERIMENT SUMMARY")
print("="*60)
print(f"\nTotal combinations tested: {len(all_results)}")
print(f"Features tested: {list(features.keys())}")
print(f"Models tested: {list(models_config.keys())}")
print(f"\nResults saved to: results/comparison.csv")
print(f"Models saved to: models/")


EXPERIMENT SUMMARY

Total combinations tested: 6
Features tested: ['BoW', 'TF-IDF']
Models tested: ['Naive Bayes', 'Logistic Regression', 'Linear SVM']

Results saved to: results/comparison.csv
Models saved to: models/
